# Testing the knowledge package — 1: the simplest complete path

`llm_reasoner.knowledge` generates *stereotypical* hypotheses: what a kind of action
involves in general, with no scene, no objects and no robot. This notebook walks the
whole path once, for one action, and checks each stage before spending the next one.

```
action -> module selection -> schema -> a real model -> fragment graph
       -> acceptance constraints -> a record -> the resolver's questions
```

**A real model is used throughout.** Nothing here is mocked: the point is to find out
what the package does with an answer it did not write.

**Read the prompt before you send it.** Generation costs roughly fifteen thousand
input tokens per call, and every stage before the call is free. Most problems show up
in the schema or the prompt, where they cost nothing to find.

## Setup

Only `ANTHROPIC_API_KEY` is needed. Everything else is in the package.

In [1]:
import json
import os
from dotenv import load_dotenv

from llm_reasoner.knowledge.microworlds.flanagan.core import \
    ActionPhase

load_dotenv(override=True)
from llm_reasoner.clients import AnthropicClient, OpenAIClient
from llm_reasoner.knowledge import KnowledgeService
from llm_reasoner.knowledge.generation import (
    CachingGenerator,
    HypothesisGenerator,
    RepairBudgetExhausted,
    derive_schema,
    to_json_schema,
)
from llm_reasoner.knowledge.hypothesis import InMemoryStore
from llm_reasoner.knowledge.microworlds import flanagan
from llm_reasoner.knowledge.microworlds.registry import (
    current_versions,
    select_modules,
    task_reference,
)
from llm_reasoner.knowledge.ontological.validation import report as validate_declarations


from dataclasses import replace

from llm_reasoner.knowledge.ontological import ConstraintSuite


SOMA = "http://www.ease-crc.org/ont/SOMA.owl#"

assert os.environ.get("ANTHROPIC_API_KEY"), "set ANTHROPIC_API_KEY before running"
assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY before running"
print("microworld loaded:", flanagan.MODULE_SCOPE)

microworld loaded: llm_reasoner.knowledge.microworlds.flanagan


## Part 1 — What the package knows, before any model is involved

Three checks, all free. If any of them is wrong, generation cannot be right.

### 1.1 Are the declarations coherent?

Structural checks over the model itself: anti-rigid classes realised as roles,
disjointness not contradicting inheritance, every field ranging over something
declared, description-level classes not reaching into occurrence-level content.

These replace what a description-logic reasoner used to catch. Zero errors is the
only acceptable result.

In [2]:
print(validate_declarations(flanagan.MODULE_SCOPE))

98 declared classes | 0 errors, 0 warnings


### 1.2 Which modules does this action need?

Selection is by **capability**, not by name. A profile says pouring presupposes phase
structuring, prehensile manipulation and substance transfer; modules advertise what
they provide; the loader matches and closes over dependencies.

The third row is the one that matters. `Decanting` is an action nothing has ever been
authored for — it inherits the right modules purely from being declared under
`soma:Pouring`.

In [3]:
actions = {
    "pour into":  task_reference(SOMA + "PouringInto",
                                 {SOMA + "Pouring", SOMA + "Actuating", SOMA + "PhysicalTask"}),
    "grasp":      task_reference(SOMA + "Grasping",
                                 {SOMA + "Manipulating", SOMA + "PhysicalTask"}),
    "decant (unseen)": task_reference("http://example.org/local#Decanting",
                                 {SOMA + "Pouring", SOMA + "Actuating", SOMA + "PhysicalTask"}),
    "wipe (no profile)": task_reference(SOMA + "Wiping", {SOMA + "PhysicalTask"}),
}

for label, action in actions.items():
    chosen = select_modules(action)
    modules = [m.name.rsplit(".", 1)[-1] for m in chosen.modules]
    flag = "  [DEGRADED — no action-specific vocabulary]" if chosen.degraded else ""
    print(f"{label:20} {str(modules):26} profile={chosen.profile.task}{flag}")

pour into            ['core', 'pouring']        profile=Pouring
grasp                ['core', 'pouring']        profile=Grasping
decant (unseen)      ['core', 'pouring']        profile=Pouring
wipe (no profile)    ['core']                   profile=PhysicalTask  [DEGRADED — no action-specific vocabulary]


### 1.3 What may the model actually say?

The output shape is derived from the compiled classes, never hand-written — so a field
renamed in the microworld changes the schema with no second edit.

Worth checking `ActionPhase`: its required fields are exactly the ones that make a
phase a phase rather than a label, and `coupling_regime` offers a closed set.

In [4]:
task = actions["pour into"]
selection = select_modules(task)

schema = derive_schema([m.name for m in selection.modules],
                       root="PouringPhaseStructure")

print("types the model may emit:", len(schema.classes))
print("root:", schema.root)
print()

phase = schema.get("ActionPhase")
print("ActionPhase")
for f in phase.fields:
    need = f"required x{f.required}" if f.required else "optional"
    opts = f"  options: {list(f.options)[:4]}{'...' if len(f.options) > 4 else ''}" if f.options else ""
    print(f"   {f.name:26} {f.kind.value:9} {need:12}{opts}")

types the model may emit: 80
root: PouringPhaseStructure

ActionPhase
   aims_for                   fragment  required x1   options: ['Subgoal']
   coupling_regime            value     required x1   options: ['UNCOUPLED', 'TRANSITIONAL', 'COUPLED']
   control_regime             value     required x1   options: ['DISCRETE_EVENT', 'CONTINUOUS']
   initiated_by               fragment  optional      options: ['ContactBreak', 'ContactEstablishment', 'FlowCessation', 'FlowOnset']...
   terminated_by              fragment  required x1   options: ['ContactBreak', 'ContactEstablishment', 'FlowCessation', 'FlowOnset']...
   phase_involves             fragment  required x1   options: ['RoleSlot']
   has_condition_requirement  fragment  optional      options: ['ConditionRequirement']
   monitors_signal            fragment  optional      options: ['AuditorySignal', 'ProprioceptiveSignal', 'TactileSignal', 'VisualSignal']
   governed_by                fragment  optional      options: ['ForwardModel'

A field ranging over an abstract base still offers its concrete kinds — traversal
passes *through* classes that cannot themselves be instantiated. Without this the
model would be told to reference a type it was never shown.

In [5]:
structure = schema.get("PouringPhaseStructure")
parameters = next(f for f in structure.fields if f.name == "defines_parameter")
print("defines_parameter accepts", len(parameters.options), "concrete kinds, e.g.")
print("  ", ", ".join(parameters.options[:8]))

defines_parameter accepts 20 concrete kinds, e.g.
   ExpectedCenterOfMass, ExpectedCenterOfMassShift, ExpectedFillLevel, ExpectedFriction, ExpectedShape, ExpectedSloshDynamics, ExpectedViscosity, ExpectedWeight


## Part 2 — The prompt, before spending a call

Two things go to the model: the prompt, and a JSON Schema constraining the response.

The schema removes the failure repair handles *worst* — an invented type name or a
misspelled field, which a model rarely learns from being told. What it cannot enforce
is anything that makes a hypothesis coherent: that a reference resolves, that a gate
ends the phase it leaves. Those are the acceptance constraints, checked after.

In [6]:
play_suite = ConstraintSuite(
      name="flanagan-play",
      constraints=()
  )

# play_suite = ConstraintSuite(
#       name="flanagan-play",
#       constraints=tuple(
#           replace(constraint, severity="warning")
#           for constraint in flanagan.CONSTRAINTS
#       ),
#   )

generator = HypothesisGenerator(
    client=AnthropicClient(),
    suite=play_suite,
    root_class="PouringPhaseStructure",
    max_repairs=4,
)

# generator = HypothesisGenerator(
#     client=AnthropicClient(),
#     suite=flanagan.SUITE,
#     root_class="PouringPhaseStructure",
#     max_repairs=4,
# )

prompt = generator.describe(
    task,
    action_label="pour cereal from cereal box into bowl",
    selection=selection,
)
json_schema = to_json_schema(schema)

print(f"prompt      {len(prompt):>7,} chars  (~{len(prompt)//4:,} tokens)")
print(f"json schema {len(json.dumps(json_schema)):>7,} chars")
print(f"client      {generator.client.identity}")

prompt       64,869 chars  (~16,217 tokens)
json schema  35,976 chars
client      anthropic:claude-haiku-4-5


Read it. Every stage after this is expensive, and most of what goes wrong is visible
here.

In [7]:
print(prompt[:3000])
print("\n   ...\n")
print(prompt[-1500:])

Describe what is involved in: pour cereal from cereal box into bowl

Produce exactly one fragment of type PouringPhaseStructure, plus every fragment it refers to, directly or indirectly.

## Ontology classification

- Select the most specific available type for every fragment.
- The fragment `type` states what the fragment is. `label` does not create a subtype.
- "label" is only a human-readable description of that typed fragment.
- Do not encode a new concept only in a label. If the vocabulary contains the concept, use its named type or concept option.

## The fragment you are building toward

PouringPhaseStructure  (a kind of ActionPhaseStructure)
  "A Flanagan phase decomposition of a pouring task."
    decomposes_task: an IRI or bare name of a term from the reference vocabulary   (required, at least 1)
      "The overall task this decomposes, as a reference into SOMA. Exactly one: this is what makes the hypothesis retrievable by action type."
    defines_phase: array of [one of: Al

## Part 3 — One real generation

The loop: ask, decode the fragment graph, wire references by id, run the microworld's
acceptance constraints, and on failure re-ask with the violated constraint quoted back
in the ontology's own vocabulary.

A hypothesis that fails repeatedly is **kept as refuted**, not discarded — a constraint
a competent model keeps violating is often a constraint saying something the vocabulary
does not let it say.

If this raises `RepairBudgetExhausted`, the cell below shows each attempt's raw
response rather than just the parse error. A response that ends mid-key or mid-value on
every attempt is truncation by an output token limit, not a formatting mistake — the
schema alone is tens of thousands of characters of instruction, and a hypothesis this
size needs real headroom in the response budget on top of it. Check `max_tokens` on
`AnthropicClient` first if this happens.

In [8]:
try:
    result = generator.generate(
        task,
        action_label="pour cereal from cereal box into bowl",
        selection=selection,
    )
    record = result.record
    print("admitted   :", result.admitted)
    print("status     :", record.status)
    print("fragments  :", len(record.scope))
    print("attempts   :", len(result.exchanges), f"({record.validation.repair_attempts} repair(s))")
    print("errors     :", len(record.validation.errors))
    print("warnings   :", len(record.validation.warnings))
    print("generator  :", record.provenance.generator)
    print("modules    :", {k.rsplit(".", 1)[-1]: v for k, v in record.provenance.module_versions.items()})
except RepairBudgetExhausted as error:
    print("failed after", len(error.exchanges), "attempt(s):", error)
    print()
    for i, exchange in enumerate(error.exchanges):
        r = exchange.raw_response
        print(f"attempt {i}: {len(r)} chars")
        print("  tail:", repr(r[-120:]))
        # a response that ends mid-token or mid-key is truncation, not a
        # formatting mistake -- check finish_reason on the client if it
        # exposes one, or raise max_tokens on AnthropicClient and retry
    print()
    print("(same data is on generator.log if you did not want to catch the error)")


admitted   : True
status     : validated
fragments  : 118
attempts   : 1 (0 repair(s))
errors     : 0
warnings   : 0
generator  : anthropic:claude-haiku-4-5
modules    : {'core': '0.2.0', 'pouring': '0.2.0'}


If it needed repairs, this is what the model was told. The value of the acceptance
suite is that it names both the fault and the fragment — a parse error can do neither.

In [9]:
print("repair_attempts reported:", record.validation.repair_attempts,
      "| exchanges actually recorded:", len(result.exchanges))
print()
print("NOTE: the prompt accumulates -- each exchange's prompt contains every")
print("prior rejection concatenated, oldest first. The LAST occurrence of a")
print("marker is what actually happened just before this exchange; the first")
print("occurrence is stale history from an earlier attempt.")
print()
for i, exchange in enumerate(result.exchanges):
    print(f"--- exchange {i} (attempt {exchange.attempt}) ---")
    print("  raw_response length:", len(exchange.raw_response))
    validation_marker = "These specific problems were found"
    reason_marker = "Reason: "
    v_idx = exchange.user.rfind(validation_marker)
    r_idx = exchange.user.rfind(reason_marker)
    if v_idx == -1 and r_idx == -1:
        print("  first attempt, no prior failure to repair")
        continue
    # whichever marker occurs LATER in the accumulated prompt is the one that
    # actually describes the failure immediately preceding this exchange
    if v_idx > r_idx:
        print("  most recent prior failure was VALIDATION -- violation list:")
        print(" ", exchange.user[v_idx:][:800].replace("\n", "\n  "))
    else:
        reason_line = exchange.user[r_idx:].splitlines()[0]
        print("  most recent prior failure was a DECODE failure:")
        print("   ", reason_line)

repair_attempts reported: 0 | exchanges actually recorded: 1

NOTE: the prompt accumulates -- each exchange's prompt contains every
prior rejection concatenated, oldest first. The LAST occurrence of a
marker is what actually happened just before this exchange; the first
occurrence is stale history from an earlier attempt.

--- exchange 0 (attempt 0) ---
  raw_response length: 24120
  first attempt, no prior failure to repair


Warnings do not block admission. They flag content that is structurally legal but
probably not what was meant — two phases the controller cannot tell apart, a margin
over an unrelated threshold. Worth reading even on a clean run.

In [21]:
result.__dict__['record']

HypothesisRecord(action_type=ConceptReference(iri='http://www.ease-crc.org/ont/SOMA.owl#PouringInto', label='PouringInto', ancestors=frozenset({'http://www.ease-crc.org/ont/SOMA.owl#Actuating', 'http://www.ease-crc.org/ont/SOMA.owl#PhysicalTask', 'http://www.ease-crc.org/ont/SOMA.owl#Pouring'})), scope=KnowledgeScope(label='', registry=<llm_reasoner.knowledge.ontological.decorators.OntologyRegistry object at 0x757dc513d700>), provenance=Provenance(generator='anthropic:claude-haiku-4-5', prompt_version='generation/v2', module_versions={'llm_reasoner.knowledge.microworlds.flanagan.core': '0.2.0', 'llm_reasoner.knowledge.microworlds.flanagan.pouring': '0.2.0'}, generated_at=datetime.datetime(2026, 8, 26, 10, 28, 14, 652616, tzinfo=datetime.timezone.utc), degraded=False, exchanges=(Exchange(system='You describe what a kind of action involves, in general.\n\nWhat you are producing:\n- A hypothesis about how a kind of manipulation is organised: its phases, what\n  each phase is for, what mar

In [10]:
for w in record.validation.warnings:
    print(" -", w)
if not record.validation.warnings:
    print("none")

none


### Visualization-only Minsky action frame overlay

This branch is independent of the Flanagan phase decomposition. It presents only the
action stereotype requested here: the action to perform, the entities generally
involved, and the expected outcomes.

Each entity unfolds into affordances, dispositions, visual properties, and geometric
properties. These are visualization-only prototype descriptions; they do not enter the
hypothesis record, its scope, validation, serialization, or EQL domain.

Each entity frame is connected by `participates_as` to its corresponding Flanagan
role slot. The shared participant is therefore explicit while frame properties and
Flanagan control relationships remain separate.

In [11]:
from llm_reasoner.knowledge.hypothesis.graph import (
      HypothesisGraphProjector,
      HypothesisGraphVisualizer,
  )

projection = HypothesisGraphProjector().project(
    view.record,
    instruction="pouring a liquid from one vessel into another",
)

visualizer = HypothesisGraphVisualizer.for_graph(projection)
visualizer.run()


NameError: name 'view' is not defined

In [12]:
from dataclasses import dataclass
from enum import StrEnum

from llm_reasoner.knowledge.hypothesis.graph import (
    HypothesisGraph,
    HypothesisGraphNode,
    HypothesisGraphNodeKind,
    HypothesisGraphProjector,
    HypothesisGraphRelation,
    HypothesisGraphVisualizer,
)
from llm_reasoner.knowledge.microworlds.flanagan.core import RoleSlot


# %% Frame vocabulary


class MinskyActionSlot(StrEnum):
    """Direct slots of the action frame."""

    ACTION = "Action"
    ENTITIES = "Entities"
    EXPECTED_OUTCOMES = "Expected Outcomes"


class EntityPropertyKind(StrEnum):
    """Property groups used to describe a stereotypical entity."""

    AFFORDANCES = "Affordances"
    DISPOSITIONS = "Dispositions"
    VISUAL_PROPERTIES = "Visual Properties"
    GEOMETRIC_PROPERTIES = "Geometric Properties"


class MinskyFrameRelation(StrEnum):
    """Relationships introduced only by the frame visualization."""

    INTERPRETED_AS = "interpreted_as"
    HAS_SLOT = "has_slot"
    HAS_VALUE = "has_value"
    HAS_ENTITY = "has_entity"
    DESCRIBED_BY = "described_by"
    HAS_PROPERTY = "has_property"
    PARTICIPATES_AS = "participates_as"


@dataclass(frozen=True)
class EntityPropertyFrame:
    """A group of stereotypical properties for an entity."""

    kind: EntityPropertyKind
    """Kind of entity property represented by the group."""

    values: tuple[str, ...]
    """Prototype property descriptions in the group."""


@dataclass(frozen=True)
class EntityFrame:
    """A stereotypical entity involved in an action."""

    label: str
    """Entity name shown in the action frame."""

    property_frames: tuple[EntityPropertyFrame, ...]
    """Affordance, disposition, visual, and geometric descriptions."""

    role_type_names: tuple[str, ...]
    """Flanagan role types that this entity typically fills."""


@dataclass(frozen=True)
class MinskyActionFrameTemplate:
    """The object-centered content of a stereotypical action frame."""

    label: str
    """Name shown on the action-frame node."""

    action: str
    """Canonical action represented by the frame."""

    entities: tuple[EntityFrame, ...]
    """Entities generally involved in the action."""

    expected_outcomes: tuple[str, ...]
    """States normally expected after successful performance."""

    def entity_for_role(self, role: RoleSlot) -> EntityFrame | None:
        """Return the entity frame corresponding to a Flanagan role slot."""
        if role.binds_as is None:
            return None
        role_type_name = role.binds_as.iri.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
        for entity in self.entities:
            if role_type_name in entity.role_type_names:
                return entity
        return None

    @classmethod
    def cereal_pouring(cls) -> "MinskyActionFrameTemplate":
        """Return an object-centered cereal-pouring stereotype."""
        return cls(
            label="Cereal-Pouring Action Frame",
            action="Pouring",
            entities=(
                EntityFrame(
                    label="Agent / Effector",
                    role_type_names=("GraspingEffector", "Actor"),
                    property_frames=(
                        EntityPropertyFrame(
                            EntityPropertyKind.AFFORDANCES,
                            (
                                "Reach and grasp the cereal box",
                                "Lift and orient the cereal box",
                                "Regulate cereal release",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.DISPOSITIONS,
                            (
                                "Moves under intentional control",
                                "Adjusts grip to the changing box load",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.VISUAL_PROPERTIES,
                            (
                                "Effector pose",
                                "Contact configuration",
                                "Visibility of the grasp",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.GEOMETRIC_PROPERTIES,
                            (
                                "Reachable workspace",
                                "Grasp span",
                                "Approach direction",
                            ),
                        ),
                    ),
                ),
                EntityFrame(
                    label="Cereal Box",
                    role_type_names=("Manipulandum", "SourceContainer"),
                    property_frames=(
                        EntityPropertyFrame(
                            EntityPropertyKind.AFFORDANCES,
                            (
                                "Graspable",
                                "Liftable",
                                "Tiltable",
                                "Supports controlled cereal pouring",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.DISPOSITIONS,
                            (
                                "Retains cereal while upright or closed",
                                "Releases cereal through its opening when tilted",
                                "May scatter cereal when misaligned",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.VISUAL_PROPERTIES,
                            (
                                "Visible package opening or top flaps",
                                "Printed rectangular package surfaces",
                                "Observable grasp region on the box panels",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.GEOMETRIC_PROPERTIES,
                            (
                                "Cuboid package dimensions",
                                "Opening size",
                                "Interior capacity",
                                "Center of mass",
                            ),
                        ),
                    ),
                ),
                EntityFrame(
                    label="Cereal",
                    role_type_names=("PouredSubstance",),
                    property_frames=(
                        EntityPropertyFrame(
                            EntityPropertyKind.AFFORDANCES,
                            (
                                "Transferable between containers",
                                "Pourable through the box opening",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.DISPOSITIONS,
                            (
                                "Flows as a granular collective under gravity",
                                "Forms a pile in the bowl",
                                "Scatters when spilled",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.VISUAL_PROPERTIES,
                            (
                                "Discrete pieces are visible",
                                "Piece color and shape are observable",
                                "A cereal stream is visible during transfer",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.GEOMETRIC_PROPERTIES,
                            (
                                "Bulk volume",
                                "Piece size and shape",
                                "Pile footprint",
                                "No fixed bulk shape",
                            ),
                        ),
                    ),
                ),
                EntityFrame(
                    label="Bowl",
                    role_type_names=("TargetReceptacle",),
                    property_frames=(
                        EntityPropertyFrame(
                            EntityPropertyKind.AFFORDANCES,
                            (
                                "Receives cereal",
                                "Contains transferred cereal",
                                "Supports portion observation",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.DISPOSITIONS,
                            (
                                "Retains cereal when stable",
                                "Fills as cereal enters",
                                "May overflow when overfilled",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.VISUAL_PROPERTIES,
                            (
                                "Visible open rim",
                                "Observable concave interior",
                                "Observable cereal level",
                            ),
                        ),
                        EntityPropertyFrame(
                            EntityPropertyKind.GEOMETRIC_PROPERTIES,
                            (
                                "Opening diameter",
                                "Concave interior",
                                "Interior capacity",
                                "Pose and orientation",
                            ),
                        ),
                    ),
                ),
            ),
            expected_outcomes=(
                "Cereal is contained by the bowl",
                "The intended cereal portion has been transferred",
                "Cereal flow has stopped",
                "No unintended cereal spill has occurred",
            ),
        )

# %% Frame overlay


@dataclass(frozen=True)
class FrameRoleLabeler:
    """Give unbound Flanagan role slots semantic display names."""

    template: MinskyActionFrameTemplate
    """Entity template used to interpret role types."""

    def label(self, node: HypothesisGraphNode) -> str:
        """Return a semantic role name or the node's original label."""
        if not isinstance(node.source, RoleSlot):
            return node.graph_label()

        entity = self.template.entity_for_role(node.source)
        if entity is not None:
            return f"{entity.label} Role"
        if node.source.label:
            return node.source.label
        if node.source.binds_as is not None and node.source.binds_as.label:
            return f"{node.source.binds_as.label} Role"
        return node.graph_label()


@dataclass(frozen=True)
class MinskyActionFrameOverlay:
    """Add an independent object-centered frame to a graph projection."""

    template: MinskyActionFrameTemplate
    """Prototype content rendered by the overlay."""

    def apply(self, projection: HypothesisGraph) -> HypothesisGraph:
        """Add the frame branch and return the same projection."""
        frame_identifier = f"minsky-frame:{projection.instruction.identifier}"
        if any(node.identifier == frame_identifier for node in projection.nodes):
            return projection

        frame_index = self._add_node(
            projection,
            identifier=frame_identifier,
            label=self.template.label,
            type_name="MinskyActionFrame",
            kind=HypothesisGraphNodeKind.ROOT,
            source=self.template,
        )
        self._connect(
            projection,
            projection.instruction_index,
            MinskyFrameRelation.INTERPRETED_AS,
            frame_index,
        )
        self._add_action_slot(projection, frame_index)
        self._add_entity_slot(projection, frame_index)
        self._add_outcome_slot(projection, frame_index)
        return projection

    def _add_action_slot(
        self,
        projection: HypothesisGraph,
        frame_index: int,
    ) -> None:
        """Add the canonical action as a direct frame slot."""
        slot_index = self._add_slot(
            projection,
            frame_index,
            MinskyActionSlot.ACTION,
            member_count=1,
        )
        value_index = self._add_node(
            projection,
            identifier=f"minsky-action:{projection.instruction.identifier}",
            label=self.template.action,
            type_name="ActionFrameValue",
            kind=HypothesisGraphNodeKind.FRAGMENT,
            source=self.template,
        )
        self._connect(
            projection,
            slot_index,
            MinskyFrameRelation.HAS_VALUE,
            value_index,
        )

    def _add_entity_slot(
        self,
        projection: HypothesisGraph,
        frame_index: int,
    ) -> None:
        """Add entity frames and their object-centered properties."""
        slot_index = self._add_slot(
            projection,
            frame_index,
            MinskyActionSlot.ENTITIES,
            member_count=len(self.template.entities),
        )
        for entity_position, entity in enumerate(self.template.entities):
            entity_index = self._add_node(
                projection,
                identifier=(
                    f"minsky-entity:{projection.instruction.identifier}:"
                    f"{entity_position}"
                ),
                label=entity.label,
                type_name="EntityFrame",
                kind=HypothesisGraphNodeKind.FRAGMENT,
                source=entity,
            )
            self._connect(
                projection,
                slot_index,
                MinskyFrameRelation.HAS_ENTITY,
                entity_index,
            )
            for role_index in self._role_node_indices(projection, entity):
                self._connect(
                    projection,
                    entity_index,
                    MinskyFrameRelation.PARTICIPATES_AS,
                    role_index,
                )
            self._add_entity_properties(
                projection,
                entity_index,
                entity_position,
                entity,
            )

    def _role_node_indices(
        self,
        projection: HypothesisGraph,
        entity: EntityFrame,
    ) -> tuple[int, ...]:
        """Return projected Flanagan role slots corresponding to an entity."""
        matched_indices: list[int] = []
        for node_index in projection.graph.node_indices():
            node_source = projection.graph[node_index].source
            if not isinstance(node_source, RoleSlot):
                continue
            if self.template.entity_for_role(node_source) is entity:
                matched_indices.append(node_index)
        return tuple(matched_indices)

    def _add_entity_properties(
        self,
        projection: HypothesisGraph,
        entity_index: int,
        entity_position: int,
        entity: EntityFrame,
    ) -> None:
        """Add the property groups and values of one entity frame."""
        for property_position, property_frame in enumerate(entity.property_frames):
            property_index = self._add_node(
                projection,
                identifier=(
                    f"minsky-property-group:{projection.instruction.identifier}:"
                    f"{entity_position}:{property_position}"
                ),
                label=(
                    f"{property_frame.kind.value} "
                    f"({len(property_frame.values)})"
                ),
                type_name="EntityPropertyGroup",
                kind=HypothesisGraphNodeKind.GROUP,
                source=property_frame,
            )
            self._connect(
                projection,
                entity_index,
                MinskyFrameRelation.DESCRIBED_BY,
                property_index,
            )
            for value_position, value in enumerate(property_frame.values):
                value_index = self._add_node(
                    projection,
                    identifier=(
                        f"minsky-property:{projection.instruction.identifier}:"
                        f"{entity_position}:{property_position}:{value_position}"
                    ),
                    label=value,
                    type_name="EntityProperty",
                    kind=HypothesisGraphNodeKind.FRAGMENT,
                    source=property_frame,
                )
                self._connect(
                    projection,
                    property_index,
                    MinskyFrameRelation.HAS_PROPERTY,
                    value_index,
                )

    def _add_outcome_slot(
        self,
        projection: HypothesisGraph,
        frame_index: int,
    ) -> None:
        """Add expected successful outcomes as a direct frame slot."""
        slot_index = self._add_slot(
            projection,
            frame_index,
            MinskyActionSlot.EXPECTED_OUTCOMES,
            member_count=len(self.template.expected_outcomes),
        )
        for outcome_position, outcome in enumerate(self.template.expected_outcomes):
            outcome_index = self._add_node(
                projection,
                identifier=(
                    f"minsky-outcome:{projection.instruction.identifier}:"
                    f"{outcome_position}"
                ),
                label=outcome,
                type_name="ExpectedOutcome",
                kind=HypothesisGraphNodeKind.FRAGMENT,
                source=self.template,
            )
            self._connect(
                projection,
                slot_index,
                MinskyFrameRelation.HAS_VALUE,
                outcome_index,
            )

    def _add_slot(
        self,
        projection: HypothesisGraph,
        frame_index: int,
        slot: MinskyActionSlot,
        member_count: int,
    ) -> int:
        """Add one direct slot beneath the action frame."""
        slot_index = self._add_node(
            projection,
            identifier=(
                f"minsky-slot:{projection.instruction.identifier}:"
                f"{slot.name.lower()}"
            ),
            label=f"{slot.value} ({member_count})",
            type_name="MinskyActionSlot",
            kind=HypothesisGraphNodeKind.GROUP,
            source=slot,
        )
        self._connect(
            projection,
            frame_index,
            MinskyFrameRelation.HAS_SLOT,
            slot_index,
        )
        return slot_index

    @staticmethod
    def _add_node(
        projection: HypothesisGraph,
        identifier: str,
        label: str,
        type_name: str,
        kind: HypothesisGraphNodeKind,
        source: object,
    ) -> int:
        """Add one synthetic frame node."""
        return projection.graph.add_node(
            HypothesisGraphNode(
                identifier=identifier,
                label=label,
                type_name=type_name,
                kind=kind,
                source=source,
            )
        )

    @staticmethod
    def _connect(
        projection: HypothesisGraph,
        source_index: int,
        relation: MinskyFrameRelation,
        target_index: int,
    ) -> None:
        """Add one visualization-only frame relationship."""
        projection.graph.add_edge(
            source_index,
            target_index,
            HypothesisGraphRelation(
                source=projection.graph[source_index],
                predicate=relation.value,
                target=projection.graph[target_index],
            ),
        )

### KRR-grounded image-schema profile

This visualization follows the ontology-pattern style used by
[ImageSchemaNet](https://www.semantic-web-journal.net/system/files/swj3084.pdf)
and its ISAAC foundation:

- an image-schema profile collects the schemas activated by an instruction or event;
- each image schema is represented as a gestalt-like conceptual frame;
- spatial primitives are reified semantic roles whose fillers are existing entities or
  schematic spatial regions;
- schema instances are distinguished from reusable schema types.

The cereal transfer is represented as a compact structured combination:
initial CONTAINMENT, SOURCE-PATH-GOAL transfer, and result CONTAINMENT.
The ordering follows work on
[image-schema combinations for complex events](https://link.springer.com/article/10.1007/s13218-019-00605-1),
while the SOURCE-PATH-GOAL roles follow the
[formal PATH-family microtheories](https://www.inf.unibz.it/~okutz/resources/Choosing-the-Right-Path_-Image-Schema-Theory-as-a-Foundation-for-Concept-Invention-%28JAGI%29.pdf).

This remains a visualization-only ABox-style projection. It reuses the Cereal Box,
Cereal, and Bowl nodes from the action frame and does not modify the hypothesis record,
validation, serialization, or EQL domain.

In [13]:
from __future__ import annotations

from dataclasses import dataclass
from enum import StrEnum

from llm_reasoner.knowledge.hypothesis.graph import (
    HypothesisGraph,
    HypothesisGraphNode,
    HypothesisGraphNodeKind,
    HypothesisGraphRelation,
)


# %% Image-schema vocabulary


class ImageSchemaKind(StrEnum):
    """Reusable image-schema types represented by this profile."""

    CONTAINMENT = "CONTAINMENT"
    SOURCE_PATH_GOAL = "SOURCE-PATH-GOAL"


class SpatialPrimitiveKind(StrEnum):
    """Semantic roles internal to the represented image schemas."""

    CONTAINER = "CONTAINER"
    CONTAINED = "CONTAINED"
    INTERIOR = "INTERIOR"
    BOUNDARY = "BOUNDARY"
    EXTERIOR = "EXTERIOR"
    TRAJECTOR = "TRAJECTOR"
    SOURCE = "SOURCE"
    PATH = "PATH"
    GOAL = "GOAL"
    DIRECTION = "DIRECTION"


class ImageSchemaFillerKind(StrEnum):
    """Ontological category of a spatial-primitive filler."""

    ENTITY = "EntityFrame"
    SPATIAL_REGION = "SpatialRegion"
    SPATIAL_PATH = "SpatialPath"
    SPATIAL_DIRECTION = "SpatialDirection"


class ImageSchemaRelation(StrEnum):
    """Relationships introduced by the image-schema visualization."""

    HAS_PROFILE = "has_image_schema_profile"
    HAS_SCHEMA_INSTANCE = "has_schema_instance"
    INSTANCE_OF = "instance_of"
    NECESSARY_SPATIAL_PRIMITIVE = "necessary_spatial_primitive"
    FILLED_BY = "filled_by"
    PRECEDES = "precedes"


class ImageSchemaGroundingError(ValueError):
    """Raised when an entity-valued primitive has no frame entity filler."""


@dataclass(frozen=True)
class SpatialPrimitiveBinding:
    """Bind one spatial-primitive role to a participant or spatial value."""

    primitive: SpatialPrimitiveKind
    """Spatial-primitive role played within an image schema."""

    filler_label: str
    """Label of the entity or spatial value filling the role."""

    filler_kind: ImageSchemaFillerKind
    """Ontological category assigned to the filler."""


@dataclass(frozen=True)
class ImageSchemaInstance:
    """A situation-specific occurrence of a reusable image schema."""

    label: str
    """Situation-specific name shown in the graph."""

    schema_kind: ImageSchemaKind
    """Reusable image-schema type instantiated by this occurrence."""

    primitive_bindings: tuple[SpatialPrimitiveBinding, ...]
    """Required spatial-primitive roles and their fillers."""


@dataclass(frozen=True)
class ImageSchemaProfile:
    """The ordered image-schematic characterization of an instruction."""

    label: str
    """Name shown on the profile node."""

    schema_instances: tuple[ImageSchemaInstance, ...]
    """Image-schema occurrences activated by the instruction."""

    @classmethod
    def cereal_pouring(cls) -> ImageSchemaProfile:
        """Return the image-schema profile for pouring cereal into a bowl."""
        return cls(
            label="Cereal-Pouring Image-Schema Profile",
            schema_instances=(
                ImageSchemaInstance(
                    label="Initial Containment",
                    schema_kind=ImageSchemaKind.CONTAINMENT,
                    primitive_bindings=(
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.CONTAINER,
                            "Cereal Box",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.CONTAINED,
                            "Cereal",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.INTERIOR,
                            "Cereal Box Interior",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.BOUNDARY,
                            "Cereal Box Boundary",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.EXTERIOR,
                            "Outside Cereal Box",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                    ),
                ),
                ImageSchemaInstance(
                    label="Cereal Transfer",
                    schema_kind=ImageSchemaKind.SOURCE_PATH_GOAL,
                    primitive_bindings=(
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.TRAJECTOR,
                            "Cereal",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.SOURCE,
                            "Cereal Box",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.PATH,
                            "Downward Transfer Path",
                            ImageSchemaFillerKind.SPATIAL_PATH,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.GOAL,
                            "Bowl",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.DIRECTION,
                            "Downward",
                            ImageSchemaFillerKind.SPATIAL_DIRECTION,
                        ),
                    ),
                ),
                ImageSchemaInstance(
                    label="Result Containment",
                    schema_kind=ImageSchemaKind.CONTAINMENT,
                    primitive_bindings=(
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.CONTAINER,
                            "Bowl",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.CONTAINED,
                            "Cereal",
                            ImageSchemaFillerKind.ENTITY,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.INTERIOR,
                            "Bowl Interior",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.BOUNDARY,
                            "Bowl Boundary",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                        SpatialPrimitiveBinding(
                            SpatialPrimitiveKind.EXTERIOR,
                            "Outside Bowl",
                            ImageSchemaFillerKind.SPATIAL_REGION,
                        ),
                    ),
                ),
            ),
        )


# %% Image-schema graph projection


@dataclass(frozen=True)
class ImageSchemaProfileOverlay:
    """Add an image-schema profile without changing the hypothesis record."""

    profile: ImageSchemaProfile
    """Situation-specific image-schema profile to project."""

    def apply(self, projection: HypothesisGraph) -> HypothesisGraph:
        """Add the image-schema branch and return the same projection."""
        profile_identifier = (
            f"image-schema-profile:{projection.instruction.identifier}"
        )
        if any(node.identifier == profile_identifier for node in projection.nodes):
            return projection

        profile_index = self._add_node(
            projection,
            identifier=profile_identifier,
            label=f"{self.profile.label} ({len(self.profile.schema_instances)})",
            type_name="ImageSchemaProfile",
            kind=HypothesisGraphNodeKind.ROOT,
            source=self.profile,
        )
        self._connect(
            projection,
            projection.instruction_index,
            ImageSchemaRelation.HAS_PROFILE,
            profile_index,
        )

        entity_indices = self._entity_indices(projection)
        filler_indices = dict(entity_indices)
        schema_type_indices: dict[ImageSchemaKind, int] = {}
        instance_indices: list[int] = []
        for instance_position, schema_instance in enumerate(
            self.profile.schema_instances
        ):
            instance_index = self._add_schema_instance(
                projection,
                profile_index,
                instance_position,
                schema_instance,
            )
            instance_indices.append(instance_index)
            schema_type_index = self._schema_type_index(
                projection,
                schema_instance.schema_kind,
                schema_type_indices,
            )
            self._connect(
                projection,
                instance_index,
                ImageSchemaRelation.INSTANCE_OF,
                schema_type_index,
            )
            self._add_spatial_primitives(
                projection,
                instance_index,
                instance_position,
                schema_instance,
                filler_indices,
            )

        for earlier_index, later_index in zip(
            instance_indices,
            instance_indices[1:],
            strict=False,
        ):
            self._connect(
                projection,
                earlier_index,
                ImageSchemaRelation.PRECEDES,
                later_index,
            )
        return projection

    def _add_schema_instance(
        self,
        projection: HypothesisGraph,
        profile_index: int,
        instance_position: int,
        schema_instance: ImageSchemaInstance,
    ) -> int:
        """Add one situation-specific image-schema occurrence."""
        instance_index = self._add_node(
            projection,
            identifier=(
                f"image-schema-instance:{projection.instruction.identifier}:"
                f"{instance_position}"
            ),
            label=schema_instance.label,
            type_name="ImageSchemaInstance",
            kind=HypothesisGraphNodeKind.GROUP,
            source=schema_instance,
        )
        self._connect(
            projection,
            profile_index,
            ImageSchemaRelation.HAS_SCHEMA_INSTANCE,
            instance_index,
        )
        return instance_index

    def _schema_type_index(
        self,
        projection: HypothesisGraph,
        schema_kind: ImageSchemaKind,
        schema_type_indices: dict[ImageSchemaKind, int],
    ) -> int:
        """Return the shared node for a reusable image-schema type."""
        if schema_kind in schema_type_indices:
            return schema_type_indices[schema_kind]

        schema_type_index = self._add_node(
            projection,
            identifier=f"image-schema-type:{schema_kind.name.lower()}",
            label=schema_kind.value,
            type_name="ImageSchema",
            kind=HypothesisGraphNodeKind.FRAGMENT,
            source=schema_kind,
        )
        schema_type_indices[schema_kind] = schema_type_index
        return schema_type_index

    def _add_spatial_primitives(
        self,
        projection: HypothesisGraph,
        instance_index: int,
        instance_position: int,
        schema_instance: ImageSchemaInstance,
        filler_indices: dict[str, int],
    ) -> None:
        """Add semantic-role nodes and bind them to shared fillers."""
        for primitive_position, binding in enumerate(
            schema_instance.primitive_bindings
        ):
            primitive_index = self._add_node(
                projection,
                identifier=(
                    f"spatial-primitive:{projection.instruction.identifier}:"
                    f"{instance_position}:{primitive_position}"
                ),
                label=binding.primitive.value,
                type_name="SpatialPrimitive",
                kind=HypothesisGraphNodeKind.GROUP,
                source=binding,
            )
            self._connect(
                projection,
                instance_index,
                ImageSchemaRelation.NECESSARY_SPATIAL_PRIMITIVE,
                primitive_index,
            )
            filler_index = self._filler_index(
                projection,
                binding,
                filler_indices,
            )
            self._connect(
                projection,
                primitive_index,
                ImageSchemaRelation.FILLED_BY,
                filler_index,
            )

    def _filler_index(
        self,
        projection: HypothesisGraph,
        binding: SpatialPrimitiveBinding,
        filler_indices: dict[str, int],
    ) -> int:
        """Return a shared entity or synthetic spatial-value node."""
        if binding.filler_label in filler_indices:
            return filler_indices[binding.filler_label]
        if binding.filler_kind is ImageSchemaFillerKind.ENTITY:
            raise ImageSchemaGroundingError(
                f"No frame entity is available for {binding.filler_label!r}."
            )

        filler_index = self._add_node(
            projection,
            identifier=(
                f"image-schema-filler:{projection.instruction.identifier}:"
                f"{binding.filler_kind.name.lower()}:"
                f"{binding.filler_label.lower().replace(' ', '-')}"
            ),
            label=binding.filler_label,
            type_name=binding.filler_kind.value,
            kind=HypothesisGraphNodeKind.FRAGMENT,
            source=binding,
        )
        filler_indices[binding.filler_label] = filler_index
        return filler_index

    @staticmethod
    def _entity_indices(projection: HypothesisGraph) -> dict[str, int]:
        """Return existing action-frame entities indexed by label."""
        return {
            projection.graph[node_index].label: node_index
            for node_index in projection.graph.node_indices()
            if projection.graph[node_index].type_name == "EntityFrame"
        }

    @staticmethod
    def _add_node(
        projection: HypothesisGraph,
        identifier: str,
        label: str,
        type_name: str,
        kind: HypothesisGraphNodeKind,
        source: object,
    ) -> int:
        """Add one image-schema projection node."""
        return projection.graph.add_node(
            HypothesisGraphNode(
                identifier=identifier,
                label=label,
                type_name=type_name,
                kind=kind,
                source=source,
            )
        )

    @staticmethod
    def _connect(
        projection: HypothesisGraph,
        source_index: int,
        relation: ImageSchemaRelation,
        target_index: int,
    ) -> None:
        """Add one image-schema relationship."""
        projection.graph.add_edge(
            source_index,
            target_index,
            HypothesisGraphRelation(
                source=projection.graph[source_index],
                predicate=relation.value,
                target=projection.graph[target_index],
            ),
        )

In [14]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass

from llm_reasoner.knowledge.hypothesis.graph import HypothesisGraphNode
from llm_reasoner.knowledge.microworlds.flanagan.core import (
    ConditionMode,
    ConditionRequirement,
    MechanicalEventType,
    PhaseTransition,
    RoleSlot,
    SensorimotorControlPoint,
    Subgoal,
)
from llm_reasoner.knowledge.ontological.decorators import KnowledgeEntity


# %% Concise ontology-style graph presentation


@dataclass(frozen=True)
class OntologyStyleGraphPresentation:
    """Present graph nodes with concise ontology-style concept names."""

    role_label_getter: Callable[[HypothesisGraphNode], str]
    """Role-specific label function used for incomplete slots."""

    def label(self, node: HypothesisGraphNode) -> str:
        """Return a concise display label without changing the graph node."""
        if node.type_name == "Instruction":
            return node.graph_label()

        representation_label = self._representation_label(node.type_name)
        if representation_label is not None:
            return representation_label
        if isinstance(node.source, RoleSlot):
            return self._role_label(node, node.source)

        phase_label = self._phase_label(node.type_name)
        if phase_label is not None:
            return phase_label
        if isinstance(node.source, PhaseTransition):
            return self._transition_label(node.source)
        if isinstance(node.source, Subgoal):
            return self._subgoal_label(node.source)
        if isinstance(node.source, ConditionRequirement):
            return self._condition_requirement_label(node.source)
        if isinstance(node.source, SensorimotorControlPoint):
            return self._control_point_label(node.source)
        if isinstance(node.source, KnowledgeEntity):
            return self._knowledge_type_label(node.type_name)
        return self._presentation_label(node)

    def information(self, node: HypothesisGraphNode) -> list[str]:
        """Return node metadata with the original prose retained."""
        details = node.graph_information()
        if self.label(node) == node.graph_label():
            return details
        return [f"description: {node.graph_label()}", *details]

    @staticmethod
    def _representation_label(type_name: str) -> str | None:
        """Return the label of a top-level action representation."""
        representation_labels = {
            "MinskyActionFrame": "ActionFrame",
            "ImageSchemaProfile": "ActionImageSchema",
        }
        if type_name in representation_labels:
            return representation_labels[type_name]
        if type_name.endswith("PhaseStructure"):
            return "ActionPhases"
        return None

    def _role_label(self, node: HypothesisGraphNode, role: RoleSlot) -> str:
        """Return the ontology concept represented by an unbound role slot."""
        if role.binds_as is None:
            return self._ontology_name(self.role_label_getter(node))

        role_type_name = role.binds_as.iri.rsplit("#", 1)[-1].rsplit("/", 1)[-1]
        role_labels = {
            "Actor": "Agent",
            "GraspingEffector": "Effector",
            "Manipulandum": "SourceContainer",
            "SourceContainer": "SourceContainer",
            "TargetReceptacle": "DestinationContainer",
            "PouredSubstance": "Substance",
            "PouringLip": "PouringLip",
            "GraspSite": "GraspSite",
            "InitialSupport": "InitialSupport",
            "FinalSupport": "FinalSupport",
        }
        if role_type_name in role_labels:
            return role_labels[role_type_name]
        return role_type_name.removesuffix("Role")

    @staticmethod
    def _phase_label(type_name: str) -> str | None:
        """Return the action concept represented by a phase class."""
        phase_labels = {
            "ReachPhase": "Reach",
            "LoadPhase": "Grasp",
            "LiftPhase": "Lift",
            "HoldPhase": "Hold",
            "TransportPhase": "Transport",
            "AlignmentPhase": "Align",
            "TiltToFlowPhase": "Tilt",
            "FlowRegulationPhase": "Pour",
            "FlowTerminationPhase": "StopFlow",
            "ReturnUprightPhase": "Upright",
            "ReplacePhase": "Lower",
            "UnloadPhase": "Release",
            "ReleasePhase": "Withdraw",
        }
        if type_name in phase_labels:
            return phase_labels[type_name]
        if type_name.endswith("Phase") and type_name != "ActionPhase":
            return type_name.removesuffix("Phase")
        return None

    def _transition_label(self, transition: PhaseTransition) -> str:
        """Name a transition from its source and target phase concepts."""
        if transition.from_phase is None or transition.to_phase is None:
            return "PhaseTransition"
        source_label = self._phase_label(type(transition.from_phase).__name__)
        target_label = self._phase_label(type(transition.to_phase).__name__)
        if source_label is None or target_label is None:
            return "PhaseTransition"
        return f"{source_label}To{target_label}"

    def _subgoal_label(self, subgoal: Subgoal) -> str:
        """Name a subgoal after the event that signals its attainment."""
        if subgoal.attainment_signalled_by is None:
            return "Subgoal"
        event_label = self._event_label(subgoal.attainment_signalled_by)
        return f"{event_label}Goal"

    def _condition_requirement_label(
        self,
        requirement: ConditionRequirement,
    ) -> str:
        """Name a condition requirement by condition and temporal mode."""
        if requirement.on_condition is None:
            return "ConditionRequirement"

        condition_label = type(requirement.on_condition).__name__.removesuffix(
            "Condition"
        )
        mode_labels = {
            ConditionMode.AT_ENTRY: "Precondition",
            ConditionMode.THROUGHOUT: "Invariant",
            ConditionMode.AT_EXIT: "Postcondition",
        }
        if requirement.mode is None:
            return f"{condition_label}Requirement"
        return f"{condition_label}{mode_labels[requirement.mode]}"

    def _control_point_label(
        self,
        control_point: SensorimotorControlPoint,
    ) -> str:
        """Name a control point after the event at which it operates."""
        if control_point.at_event_type is None:
            return "ControlPoint"
        return f"{self._event_label(control_point.at_event_type)}ControlPoint"

    @staticmethod
    def _event_label(event: MechanicalEventType) -> str:
        """Return a concise event concept from its concrete ontology class."""
        return OntologyStyleGraphPresentation._knowledge_type_label(
            type(event).__name__
        )

    @staticmethod
    def _knowledge_type_label(type_name: str) -> str:
        """Shorten a concrete knowledge class name for display."""
        if type_name == "MechanicalEventType":
            return "MechanicalEvent"
        if type_name.endswith("EventType"):
            return type_name.removesuffix("EventType")
        if type_name.endswith("Event"):
            return type_name.removesuffix("Event")
        return type_name

    def _presentation_label(self, node: HypothesisGraphNode) -> str:
        """Name synthetic frame and image-schema presentation nodes."""
        ontology_named_types = {
            "ActionFrameValue",
            "EntityFrame",
            "EntityProperty",
            "EntityPropertyGroup",
            "ExpectedOutcome",
            "ImageSchema",
            "ImageSchemaInstance",
            "MinskyActionSlot",
            "RelationshipGroup",
            "SpatialDirection",
            "SpatialPath",
            "SpatialPrimitive",
            "SpatialRegion",
        }
        if node.type_name in ontology_named_types:
            return self._phrase_label(node.label)
        return node.graph_label()

    def _phrase_label(self, phrase: str) -> str:
        """Map prototype prose to its concise ontology-style concept."""
        concept_labels = {
            "Reach and grasp the cereal box": "BoxManipulation",
            "Lift and orient the cereal box": "BoxPoseControl",
            "Regulate cereal release": "FlowRegulation",
            "Moves under intentional control": "IntentionalMotion",
            "Adjusts grip to the changing box load": "GripLoadAdaptation",
            "Supports controlled cereal pouring": "ControlledPouring",
            "Retains cereal while upright or closed": "CerealRetention",
            "Releases cereal through its opening when tilted": "TiltInducedRelease",
            "May scatter cereal when misaligned": "MisalignmentSpillRisk",
            "Visible package opening or top flaps": "VisibleOpening",
            "Printed rectangular package surfaces": "RectangularSurface",
            "Observable grasp region on the box panels": "GraspRegion",
            "Transferable between containers": "Transferable",
            "Pourable through the box opening": "Pourable",
            "Flows as a granular collective under gravity": "GravityDrivenFlow",
            "Forms a pile in the bowl": "PileFormation",
            "Scatters when spilled": "SpillScattering",
            "Discrete pieces are visible": "DiscretePieces",
            "Piece color and shape are observable": "PieceAppearance",
            "A cereal stream is visible during transfer": "VisibleFlow",
            "No fixed bulk shape": "VariableBulkShape",
            "Receives cereal": "Receivable",
            "Contains transferred cereal": "Containment",
            "Supports portion observation": "PortionObservable",
            "Retains cereal when stable": "StableRetention",
            "Fills as cereal enters": "FillResponse",
            "May overflow when overfilled": "OverflowRisk",
            "Visible open rim": "VisibleRim",
            "Observable concave interior": "ConcaveInterior",
            "Observable cereal level": "CerealLevel",
            "Cereal is contained by the bowl": "CerealContained",
            "The intended cereal portion has been transferred": "PortionTransferred",
            "Cereal flow has stopped": "FlowStopped",
            "No unintended cereal spill has occurred": "SpillAvoided",
            "Initial Containment": "InitialContainment",
            "Cereal Transfer": "CerealTransfer",
            "Result Containment": "ResultContainment",
            "SOURCE-PATH-GOAL": "SourcePathGoal",
            "Cereal Box Interior": "CerealBoxInterior",
            "Cereal Box Boundary": "CerealBoxBoundary",
            "Outside Cereal Box": "CerealBoxExterior",
            "Downward Transfer Path": "TransferPath",
            "Bowl Interior": "BowlInterior",
            "Bowl Boundary": "BowlBoundary",
            "Outside Bowl": "BowlExterior",
        }
        uncounted_phrase = phrase.partition(" (")[0]
        if uncounted_phrase in concept_labels:
            return concept_labels[uncounted_phrase]
        return self._ontology_name(uncounted_phrase)

    @staticmethod
    def _ontology_name(phrase: str) -> str:
        """Convert a readable phrase to a compact PascalCase concept name."""
        separated_phrase = phrase
        for separator in ("-", "–", "/", "_", "(", ")", ":"):
            separated_phrase = separated_phrase.replace(separator, " ")
        words = [word for word in separated_phrase.split() if not word.isdecimal()]
        return "".join(
            word.title() if word.isupper() else f"{word[:1].upper()}{word[1:]}"
            for word in words
        )


In [15]:
# %% Build one projection with three complementary representations

framed_projection = HypothesisGraphProjector().project(
    record,
    instruction="Pour cereal from the cereal box into the bowl",
)
frame_template = MinskyActionFrameTemplate.cereal_pouring()
MinskyActionFrameOverlay(
    template=frame_template,
).apply(framed_projection)

image_schema_profile = ImageSchemaProfile.cereal_pouring()
ImageSchemaProfileOverlay(
    profile=image_schema_profile,
).apply(framed_projection)

frame_role_labeler = FrameRoleLabeler(template=frame_template)
ontology_style_presentation = OntologyStyleGraphPresentation(
    role_label_getter=frame_role_labeler.label,
)
frame_visualizer = HypothesisGraphVisualizer.for_graph(
    framed_projection,
    label_getter=ontology_style_presentation.label,
    information_getter=ontology_style_presentation.information,
    port=8055,
)
frame_visualizer.run()

 * Serving Flask app 'llm_reasoner.knowledge.hypothesis.graph'
 * Debug mode: off


 * Running on http://127.0.0.1:8055
Press CTRL+C to quit
127.0.0.1 - - [26/Aug/2026 12:28:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [26/Aug/2026 12:28:42] "GET /graph HTTP/1.1" 200 -
127.0.0.1 - - [26/Aug/2026 12:28:42] "GET /favicon.ico HTTP/1.1" 404 -


### Completely offline interactive export

The next cell captures every node, edge, label, detail line, style option, and
parent-child relationship from the combined graph. The exporter embeds both the graph data
and the locally installed vis-network browser library into one HTML file.

The saved page starts collapsed and implements expansion and collapse entirely in the
browser. It does not call Flask endpoints and needs neither a running notebook kernel nor an
internet connection.

In [16]:
from __future__ import annotations

import json
from dataclasses import dataclass
from html import escape
from importlib import resources
from pathlib import Path

from llm_reasoner.knowledge.hypothesis.graph import HypothesisGraphVisualizer


# %% Offline graph snapshot


class OfflineGraphExportError(RuntimeError):
    """Raised when a standalone graph cannot be produced."""


@dataclass(frozen=True)
class OfflineGraphNode:
    """A browser-independent snapshot of one graph node."""

    identifier: str
    """Identifier used by vis-network and graph relationships."""

    label: str
    """Human-readable node label."""

    details: tuple[str, ...]
    """Detail lines shown when the node is selected."""

    background_color: str
    """Fill color used for the node."""

    border_color: str
    """Border color used for the node."""

    fixed: bool
    """Whether the node remains fixed at the center."""

    def network_data(self) -> dict[str, object]:
        """Return the JSON-compatible vis-network representation."""
        data: dict[str, object] = {
            "id": self.identifier,
            "label": self.label,
            "details": self.details,
            "color": {
                "background": self.background_color,
                "border": self.border_color,
            },
            "physics": not self.fixed,
        }
        if self.fixed:
            data.update({"x": 0.0, "y": 0.0, "fixed": True})
        return data


@dataclass(frozen=True)
class OfflineGraphEdge:
    """A browser-independent snapshot of one directed relationship."""

    identifier: str
    """Identifier unique within the exported graph."""

    source_identifier: str
    """Identifier of the relationship source."""

    target_identifier: str
    """Identifier of the relationship target."""

    label: str
    """Semantic relationship label."""

    def network_data(self) -> dict[str, str]:
        """Return the JSON-compatible vis-network representation."""
        return {
            "id": self.identifier,
            "from": self.source_identifier,
            "to": self.target_identifier,
            "label": self.label,
        }


@dataclass(frozen=True)
class OfflineHypothesisGraphSnapshot:
    """All data required to reconstruct a hypothesis graph in a browser."""

    nodes: tuple[OfflineGraphNode, ...]
    """Every graph node, including nodes initially hidden by collapsing."""

    edges: tuple[OfflineGraphEdge, ...]
    """Every directed relationship in the graph."""

    center_identifier: str
    """Identifier of the instruction node fixed at the center."""

    network_options: dict[str, object]
    """Typography, edge, color, and physics options used by vis-network."""

    @classmethod
    def capture(
        cls,
        visualizer: HypothesisGraphVisualizer,
    ) -> OfflineHypothesisGraphSnapshot:
        """Capture the full graph independently of its current expansion state."""
        nodes = tuple(
            OfflineGraphNode(
                identifier=str(node_index),
                label=visualizer.node_label(node_index),
                details=tuple(
                    visualizer.information_getter(visualizer.graph[node_index])
                ),
                background_color=visualizer.node_color(node_index),
                border_color=(
                    visualizer.node_border_color(node_index)
                    or visualizer.default_border_color
                ),
                fixed=node_index == visualizer.center_index,
            )
            for node_index in visualizer.graph.node_indices()
        )
        edges = tuple(
            OfflineGraphEdge(
                identifier=str(edge_number),
                source_identifier=str(source_index),
                target_identifier=str(target_index),
                label=relation.predicate,
            )
            for edge_number, (source_index, target_index, relation) in enumerate(
                visualizer.graph.weighted_edge_list()
            )
        )
        return cls(
            nodes=nodes,
            edges=edges,
            center_identifier=str(visualizer.center_index),
            network_options=visualizer.network_options(),
        )


# %% Standalone HTML rendering


@dataclass(frozen=True)
class OfflineHypothesisGraphExporter:
    """Render a graph snapshot as a self-contained interactive HTML file."""

    snapshot: OfflineHypothesisGraphSnapshot
    """Full graph snapshot embedded in the output."""

    title: str = "Cereal-Pouring Knowledge Graph"
    """Browser-page title and visible heading."""

    def save(self, output_path: Path) -> Path:
        """Write the standalone page and return its absolute path."""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(self.render(), encoding="utf-8")
        return output_path.resolve()

    def render(self) -> str:
        """Return a complete HTML document with no external dependencies."""
        template = """<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<script>
__VIS_NETWORK_SOURCE__
</script>
<style>
  html, body {
    margin: 0;
    height: 100%;
    background: #ffffff;
    color: #202124;
    font-family: Arial, sans-serif;
    font-size: 13px;
    font-weight: 400;
    text-shadow: none;
  }

  #toolbar {
    align-items: center;
    border-bottom: 1px solid #dadce0;
    display: flex;
    gap: 0.75rem;
    height: 42px;
    padding: 0 0.8rem;
  }

  #toolbar h1 {
    font-size: 14px;
    font-weight: 400;
    margin: 0;
  }

  #toolbar button {
    background: #ffffff;
    border: 1px solid #9aa0a6;
    border-radius: 4px;
    color: #202124;
    cursor: pointer;
    font-family: Arial, sans-serif;
    font-size: 12px;
    padding: 0.3rem 0.6rem;
  }

  #offline-status {
    color: #5f6368;
    margin-left: auto;
  }

  #container {
    display: flex;
    height: calc(100vh - 43px);
  }

  #network {
    background: #ffffff;
    flex: 3;
  }

  #details {
    background: #ffffff;
    border-left: 1px solid #dadce0;
    flex: 1;
    overflow: auto;
    padding: 1rem;
  }

  #details h3 {
    font-size: 14px;
    font-weight: 400;
    margin-top: 0;
  }

  #details p {
    margin: 0.5rem 0;
  }
</style>
</head>
<body>
<div id="toolbar">
  <h1>__TITLE__</h1>
  <button id="collapse-all" type="button">Collapse all</button>
  <button id="fit-graph" type="button">Fit visible graph</button>
  <span id="offline-status">Offline standalone graph</span>
</div>
<div id="container">
  <div id="network"></div>
  <div id="details">
    <i>Click a node to expand it. Double-click it to collapse its branch.</i>
  </div>
</div>
<script>
(() => {
  const allNodes = __NODES_JSON__;
  const allEdges = __EDGES_JSON__;
  const centerIdentifier = __CENTER_JSON__;
  const networkOptions = __OPTIONS_JSON__;
  const nodeByIdentifier = new Map(allNodes.map((node) => [node.id, node]));
  const outgoing = new Map(allNodes.map((node) => [node.id, []]));
  const expandedIdentifiers = new Set();
  const visibleNodes = new vis.DataSet([]);
  const visibleEdges = new vis.DataSet([]);
  const detailsPanel = document.getElementById("details");
  let pendingClickTimeout;

  allEdges.forEach((edge) => {
    if (!outgoing.has(edge.from)) {
      outgoing.set(edge.from, []);
    }
    outgoing.get(edge.from).push(edge.to);
  });

  const network = new vis.Network(
    document.getElementById("network"),
    { nodes: visibleNodes, edges: visibleEdges },
    networkOptions
  );

  function synchronize(dataSet, incomingItems) {
    const incomingIdentifiers = new Set(incomingItems.map((item) => item.id));
    dataSet.update(incomingItems);
    dataSet.getIds().forEach((identifier) => {
      if (!incomingIdentifiers.has(identifier)) {
        dataSet.remove(identifier);
      }
    });
  }

  function visibleIdentifiers() {
    const identifiers = new Set([centerIdentifier]);
    expandedIdentifiers.forEach((identifier) => {
      identifiers.add(identifier);
      (outgoing.get(identifier) || []).forEach((child) => identifiers.add(child));
    });
    return identifiers;
  }

  function renderGraph() {
    const identifiers = visibleIdentifiers();
    const nodes = allNodes.filter((node) => identifiers.has(node.id));
    const edges = allEdges.filter(
      (edge) => expandedIdentifiers.has(edge.from) && identifiers.has(edge.to)
    );
    synchronize(visibleNodes, nodes);
    synchronize(visibleEdges, edges);
  }

  function showDetails(nodeIdentifier, message) {
    const node = nodeByIdentifier.get(nodeIdentifier);
    detailsPanel.replaceChildren();
    const heading = document.createElement("h3");
    heading.textContent = node.label;
    detailsPanel.appendChild(heading);
    node.details.forEach((line) => {
      const paragraph = document.createElement("p");
      paragraph.textContent = line;
      detailsPanel.appendChild(paragraph);
    });
    if (message) {
      const paragraph = document.createElement("p");
      paragraph.textContent = message;
      detailsPanel.appendChild(paragraph);
    }
  }

  function descendantsOf(nodeIdentifier) {
    const descendants = new Set();
    const frontier = [...(outgoing.get(nodeIdentifier) || [])];
    while (frontier.length > 0) {
      const current = frontier.pop();
      if (descendants.has(current)) {
        continue;
      }
      descendants.add(current);
      frontier.push(...(outgoing.get(current) || []));
    }
    return descendants;
  }

  function expandNode(nodeIdentifier) {
    expandedIdentifiers.add(nodeIdentifier);
    showDetails(nodeIdentifier, "");
    renderGraph();
  }

  function collapseBranch(nodeIdentifier) {
    expandedIdentifiers.delete(nodeIdentifier);
    descendantsOf(nodeIdentifier).forEach((identifier) =>
      expandedIdentifiers.delete(identifier)
    );
    showDetails(nodeIdentifier, "Branch collapsed.");
    renderGraph();
  }

  network.on("click", (parameters) => {
    if (parameters.nodes.length === 0) {
      return;
    }
    window.clearTimeout(pendingClickTimeout);
    const nodeIdentifier = parameters.nodes[0];
    pendingClickTimeout = window.setTimeout(
      () => expandNode(nodeIdentifier),
      300
    );
  });

  network.on("doubleClick", (parameters) => {
    window.clearTimeout(pendingClickTimeout);
    if (parameters.nodes.length === 0) {
      return;
    }
    collapseBranch(parameters.nodes[0]);
  });

  document.getElementById("collapse-all").addEventListener("click", () => {
    expandedIdentifiers.clear();
    detailsPanel.innerHTML =
      "<i>Click a node to expand it. Double-click it to collapse its branch.</i>";
    renderGraph();
    network.fit({ animation: true });
  });

  document.getElementById("fit-graph").addEventListener("click", () => {
    network.fit({ animation: true });
  });

  renderGraph();
})();
</script>
</body>
</html>
"""
        replacements = {
            "__TITLE__": escape(self.title),
            "__VIS_NETWORK_SOURCE__": self._vis_network_source(),
            "__NODES_JSON__": self._script_json(
                [node.network_data() for node in self.snapshot.nodes]
            ),
            "__EDGES_JSON__": self._script_json(
                [edge.network_data() for edge in self.snapshot.edges]
            ),
            "__CENTER_JSON__": self._script_json(self.snapshot.center_identifier),
            "__OPTIONS_JSON__": self._script_json(self.snapshot.network_options),
        }
        for placeholder, value in replacements.items():
            template = template.replace(placeholder, value)
        return template

    @staticmethod
    def _vis_network_source() -> str:
        """Return the installed vis-network browser bundle for embedding."""
        asset = (
            resources.files("pyvis")
            .joinpath("lib")
            .joinpath("vis-9.1.2")
            .joinpath("vis-network.min.js")
        )
        if not asset.is_file():
            raise OfflineGraphExportError(
                "pyvis does not provide the vis-network 9.1.2 browser bundle."
            )
        return asset.read_text(encoding="utf-8").replace(
            "</script",
            "<\\/script",
        )

    @staticmethod
    def _script_json(value: object) -> str:
        """Serialize data safely inside an HTML script element."""
        return json.dumps(
            value,
            ensure_ascii=False,
            separators=(",", ":"),
        ).replace("</", "<\\/")


In [17]:
from pathlib import Path


offline_graph_snapshot = OfflineHypothesisGraphSnapshot.capture(frame_visualizer)
offline_graph_path = OfflineHypothesisGraphExporter(
    snapshot=offline_graph_snapshot,
).save(
    Path("llm_reasoner/notebooks/cereal_pouring_knowledge_graph.html")
)

print("Offline graph:", offline_graph_path)
print("Nodes        :", len(offline_graph_snapshot.nodes))
print("Edges        :", len(offline_graph_snapshot.edges))
print("Size         :", f"{offline_graph_path.stat().st_size / 1_000_000:.2f} MB")

127.0.0.1 - - [26/Aug/2026 12:29:06] "GET /graph HTTP/1.1" 200 -


Offline graph: /home/malineni/VC_Pools/cognitive_robot_abstract_machine/llm_reasoner/notebooks/llm_reasoner/notebooks/cereal_pouring_knowledge_graph.html
Nodes        : 235
Edges        : 630
Size         : 0.57 MB


## Part 4 — Reading it as the resolver would

The resolver never holds a hypothesis. It asks a fixed set of questions and gets back
**cited** answers it may weigh, use partly, or reject.

`KnowledgeService` is the whole interface between the two halves. Everything below it
may change so long as these questions still have answers.

127.0.0.1 - - [26/Aug/2026 12:29:09] "GET /graph HTTP/1.1" 200 -


In [ ]:
store = InMemoryStore()
store.put(record)

service = KnowledgeService(
    store=store,
    module_versions=current_versions(),
    generator=generator,
)

view = service.view_for(task)
print("view served:", view is not None, "| degraded:", view.degraded)

### The one answer to act on rather than weigh

Everything else is structure the resolver may take or leave. An assumption is a claim
about the world that must hold *before* the hypothesis applies at all — so it is
checked, and reported when it cannot be.

In [ ]:
for a in view.assumptions():
    print(" -", a.description, f"({len(a.on_roles)} role(s) to bind first)")
if not view.assumptions():
    print("none stated — worth noticing, since most actions presuppose something")

### The rest of the questions

Note that phases come back in **execution order**, walked from the transition graph
rather than the order the model happened to emit them.

In [ ]:
print("ROLES TO BIND")
for c in view.roles():
    print("   ", c.label)

print("\nPHASES, in execution order")
for c in view.phases():
    print("   ", c.citation)

In [ ]:
phases = view.phases()
subject = phases[len(phases) // 2]

print("PHASE:", subject.label)
goal = view.subgoal_of(subject)
print("  subgoal      :", goal.label if goal else "-")
print("  ends at      :", [c.label for c in view.boundaries_of(subject)])
print("  preconditions:", [c.value.on_condition.label for c in view.preconditions_of(subject)])
print("  invariants   :", [c.value.on_condition.label for c in view.invariants_of(subject)])
print("  monitors     :", [c.label for c in view.monitored_signals(subject)])
for c in view.parameters_for(subject):
    r = c.value.parameter_range
    print(f"  parameter    : {c.label} = {r.lower_bound}–{r.upper_bound} {r.unit}")

Ranges rather than measurements is the honest generic answer. Narrowing one is the
resolver's job, from the world or from the request; a point value would be claiming an
observation the model never made.

In [ ]:
print("WHAT CAN GO WRONG")
for c in view.failure_modes():
    print("   ", c.label)

first = view.phases()[0]
for boundary in view.boundaries_of(first):
    corrective = view.corrective_for(boundary)
    if corrective:
        print(f"\nif '{boundary.label}' does not arrive as predicted:")
        for c in corrective:
            print("   ", c.label)

### Citations

Every answer carries the record and fragment it came from, so a resolution can state
what it leaned on — and the citation still resolves after the hypothesis has been
superseded or refuted, which is exactly when an old decision has to be explained.

In [ ]:
citation = view.phases()[0].citation
print("citation:", citation)
print("resolves to:", service.cited(citation))

## Part 5 — Caching, and what happens when the ontology moves

The cache key is action type plus module versions, generator and prompt version. No
episodic component: a hypothesis is generic by construction, so the situation it was
first generated for has no bearing on reuse.

A module version bump does not delete anything. The old record stops being served and
stays retrievable, because a resolution taken under it has to remain explainable.

In [ ]:
print("second lookup hits the store, no model call:",
      service.view_for(task) is not None, "| store size:", len(store))

bumped = dict(current_versions())
first_module = next(iter(bumped))
bumped[first_module] = "9.9.9"

stale_service = KnowledgeService(store=store, module_versions=bumped)
outcome = stale_service.explain(task)
print("\nafter bumping", first_module.rsplit('.', 1)[-1], "to 9.9.9")
print("  servable :", outcome.hit)
print("  stale    :", len(outcome.stale))
print("  refuted  :", len(outcome.refuted))
print("  citation still resolves:", stale_service.cited(citation) is not None)

### Generating through the cache

`KnowledgeService` consults the store on the way to answering a question.
`CachingGenerator` is the same policy for callers that want hypotheses without the
query surface — pre-generating a batch, say, or a script that only wants the record.

It also refuses to spend calls rediscovering a failure. A hypothesis refuted because
it kept violating one constraint will usually be refuted the same way by the same model
under the same prompt, so a refuted record under an unmoved cache key is returned
rather than regenerated. `regenerate_refuted=True` is the right setting after the
ontology or the prompt changes — which is when the key has moved anyway.

In [ ]:
caching = CachingGenerator(generator=generator, store=store)

again = caching.get(task, action_label="pouring a liquid from one vessel into another")
print("from_cache :", again.from_cache, "  (the store already holds one)")
print("status     :", again.record.status)
print("passed over:", len(again.lookup.stale), "stale,", len(again.lookup.refuted), "refuted")
print("store size :", len(store), "— unchanged, so no call was made")

## Part 6 — Rejection

The other half of *guidance, not solution*. A resolver that may reject needs somewhere
for the rejection to go, or the same wrong stereotype is served again tomorrow.

The record is kept rather than deleted: a hypothesis that fails in one situation may
hold in others, and the pattern of *where* it fails is the evidence that would
eventually justify an exception rule.

In [ ]:
view.reject("worth trying: state a real reason from an actual resolution attempt",
            fragment=view.phases()[0])

print("status         :", view.record.status)
print("defeaters      :", len(view.record.defeaters))
print("served again   :", service.view_for(task, generate_if_missing=False) is not None)
print("still explained:", len(service.explain(task).refuted), "refuted record(s) retained")

## What to try next

Each of these stresses a different part, and they get harder in roughly this order.

1. **The same action twice.** Two generations of `PouringInto` from one model. Do the
   phase counts agree? Do both name the same failure events? Disagreement is not a bug
   — it is the measure of how much the ontology is actually constraining.

2. **A degraded action.** `soma:Wiping` has no profile, so only the action-neutral core
   loads. It should still generate, using general types and free-text labels. If it
   fails outright, graceful degradation is not working.

3. **An unseen action under a known parent.** `Decanting` inherits the pouring profile.
   Compare it against the pouring hypothesis: how much does the action label alone
   change?

4. **An action the microworld is wrong for** — stirring, or opening a drawer. What the
   constraints reject here says more about the ontology than about the model.

5. **Repair under pressure.** Drop `max_repairs` to 0 and see how often a first answer
   is admissible. That number is the real measure of whether the prompt and schema are
   carrying their weight.

6. **A second microworld.** The point at which `ontological/` stops being reusable in
   principle and starts being reusable in fact.